# Transformer Inference for Abstract and Keyword Generation

This notebook runs the full inference pipeline for Computer Science papers:

1. load a fine-tuned transformer summarization model
2. fetch Computer Science papers from arXiv
3. generate an abstract
4. extract keywords with TF-IDF
5. compute summary and keyword metrics


In [ ]:
!pip install -q transformers datasets evaluate sentencepiece scikit-learn beautifulsoup4 lxml rouge_score

In [ ]:
import re
import requests
from bs4 import BeautifulSoup
from collections import Counter
import xml.etree.ElementTree as ET
import evaluate
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

ARXIV_CATEGORIES = ['cs.AI', 'cs.CL', 'cs.LG', 'cs.CV']
MODEL_DIR = '/content/cs_transformer_summarizer'
FALLBACK_MODEL = 'sshleifer/distilbart-cnn-12-6'
MAX_SOURCE_TOKENS = 1024
MAX_TARGET_TOKENS = 160
MAX_KEYWORDS = 8
NUM_SAMPLE_PAPERS = 5

rouge = evaluate.load('rouge')


In [ ]:
def fetch_recent_paper_links(categories, max_per_category=10):
    links = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; Project26/1.0)'
    }

    for category in categories:
        url = (
            'http://export.arxiv.org/api/query?'
            f'search_query=cat:{category}'
            f'&start=0&max_results={max_per_category}'
            '&sortBy=submittedDate&sortOrder=descending'
        )

        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        root = ET.fromstring(response.text)

        for entry in root.findall('{http://www.w3.org/2005/Atom}entry'):
            paper_id = entry.find('{http://www.w3.org/2005/Atom}id').text.strip()
            paper_id = paper_id.split('/')[-1]
            html_link = f'https://arxiv.org/html/{paper_id}'
            links.append(html_link)

    return links

def download_arxiv_html(arxiv_html_url):
    response = requests.get(arxiv_html_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for script in soup(['script', 'style']):
        script.extract()
    return soup

def extract_abstract_and_body(soup):
    abstract_div = soup.find('div', class_='ltx_abstract')
    abstract = abstract_div.get_text(' ', strip=True) if abstract_div else ''
    if abstract_div:
        abstract_div.extract()
    body_paragraphs = []
    for p in soup.find_all('p', class_='ltx_p'):
        text = p.get_text(' ', strip=True)
        text = re.sub(r'\[\s*(\d+\s*,?\s*)+\]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            body_paragraphs.append(text)
    body_text = ' '.join(body_paragraphs)
    abstract = abstract.replace('Abstract:', '').strip()
    return abstract, body_text

def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
def extract_keywords_tfidf(text, top_n=MAX_KEYWORDS):
    if not text.strip():
        return []
    sentences = [segment.strip() for segment in re.split(r'(?<=[.!?])\s+', text) if segment.strip()]
    if not sentences:
        sentences = [text]
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=2000)
    try:
        matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return []
    scores = np.asarray(matrix.sum(axis=0)).ravel()
    features = vectorizer.get_feature_names_out()
    ranked = []
    for keyword, score in zip(features, scores):
        cleaned = keyword.strip().lower()
        if len(cleaned) < 3 or cleaned.isdigit():
            continue
        ranked.append((cleaned, float(score)))
    ranked.sort(key=lambda item: item[1], reverse=True)
    return [keyword for keyword, _ in ranked[:top_n]]

def extract_reference_terms(text, top_n=MAX_KEYWORDS):
    tokens = [token.lower() for token in re.findall(r'[A-Za-z][A-Za-z0-9_-]+', text) if len(token) > 2]
    counts = Counter(tokens)
    return [token for token, _ in counts.most_common(top_n)]

def compute_set_metrics(predicted_items, reference_items):
    predicted = {item.lower().strip() for item in predicted_items if item}
    reference = {item.lower().strip() for item in reference_items if item}
    if not predicted and not reference:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
    if not predicted or not reference:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    overlap = predicted & reference
    precision = len(overlap) / len(predicted)
    recall = len(overlap) / len(reference)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'overlap': sorted(overlap)}


In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
    ACTIVE_MODEL_NAME = MODEL_DIR
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(FALLBACK_MODEL)
    ACTIVE_MODEL_NAME = FALLBACK_MODEL

print('Loaded summarization model from:', ACTIVE_MODEL_NAME)

In [ ]:
def generate_summary(source_text):
    inputs = tokenizer(source_text, return_tensors='pt', max_length=MAX_SOURCE_TOKENS, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_length=MAX_TARGET_TOKENS,
        min_length=60,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def summarize_cs_paper(link):
    soup = download_arxiv_html(link)
    abstract, body_text = extract_abstract_and_body(soup)
    body_text = clean_text(body_text)
    generated_summary = generate_summary(body_text)

    extracted_keywords = extract_keywords_tfidf(body_text)
    reference_keywords = extract_reference_terms(abstract)
    generated_summary_terms = extract_reference_terms(generated_summary)

    rouge_scores = rouge.compute(
        predictions=[generated_summary],
        references=[abstract],
        use_stemmer=True,
    ) if abstract else {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    keyword_vs_reference = compute_set_metrics(extracted_keywords, reference_keywords)
    keyword_vs_summary = compute_set_metrics(extracted_keywords, generated_summary_terms)

    return {
        'paper_link': link,
        'outputs': {
            'generated_abstract': generated_summary,
            'extracted_keywords': extracted_keywords,
        },
        'references': {
            'original_abstract': abstract,
            'reference_keywords': reference_keywords,
            'generated_abstract_terms': generated_summary_terms,
        },
        'metrics': {
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL'],
            'keyword_vs_reference': keyword_vs_reference,
            'keyword_vs_generated_abstract': keyword_vs_summary,
        },
    }

def print_result(result, index):
    print('=' * 90)
    print(f'Sample Paper {index}')
    print('Link:', result['paper_link'])
    print('-' * 90)
    print('Generated Abstract:')
    print(result['outputs']['generated_abstract'])
    print('-' * 90)
    print('Extracted Keywords:')
    print(', '.join(result['outputs']['extracted_keywords']) or 'None')
    print('-' * 90)
    print('Original Abstract:')
    print(result['references']['original_abstract'])
    print('-' * 90)
    print('Reference Keywords:', result['references']['reference_keywords'])
    print('Generated Abstract Terms:', result['references']['generated_abstract_terms'])
    print('-' * 90)
    print('ROUGE-1:', round(result['metrics']['rouge1'], 4))
    print('ROUGE-2:', round(result['metrics']['rouge2'], 4))
    print('ROUGE-L:', round(result['metrics']['rougeL'], 4))
    print('Keyword vs Reference F1:', round(result['metrics']['keyword_vs_reference']['f1'], 4))
    print('Keyword vs Generated Abstract F1:', round(result['metrics']['keyword_vs_generated_abstract']['f1'], 4))


In [ ]:
paper_links = fetch_recent_paper_links(ARXIV_CATEGORIES, max_per_category=5)
sample_links = paper_links[:NUM_SAMPLE_PAPERS]
results = []

for index, link in enumerate(sample_links, start=1):
    try:
        result = summarize_cs_paper(link)
        results.append(result)
        print_result(result, index)
    except Exception as exc:
        print(f'Failed on {link}: {exc}')

if results:
    avg_rouge1 = np.mean([item['metrics']['rouge1'] for item in results])
    avg_rouge2 = np.mean([item['metrics']['rouge2'] for item in results])
    avg_rougeL = np.mean([item['metrics']['rougeL'] for item in results])
    avg_keyword_ref_f1 = np.mean([item['metrics']['keyword_vs_reference']['f1'] for item in results])
    avg_keyword_summary_f1 = np.mean([item['metrics']['keyword_vs_generated_abstract']['f1'] for item in results])
    print('\n' + '=' * 90)
    print('VALIDATION SUMMARY FOR VIVA/DEMO')
    print('=' * 90)
    print('Average ROUGE-1:', round(avg_rouge1, 4))
    print('Average ROUGE-2:', round(avg_rouge2, 4))
    print('Average ROUGE-L:', round(avg_rougeL, 4))
    print('Average Keyword vs Reference F1:', round(avg_keyword_ref_f1, 4))
    print('Average Keyword vs Generated Abstract F1:', round(avg_keyword_summary_f1, 4))
